In [ ]:
import pandas as pd
import numpy as np
import json

# ==============================
# 1. LOAD DATA
# ==============================
# Load the cleaned dataset that feeds the training-time feature contract.
df = pd.read_csv('cleaned.csv')

print("Columns available:", df.columns.tolist())

# ==============================
# 2. VALIDATE REQUIRED COLUMNS
# ==============================
# Keep the required schema explicit so feature engineering fails fast on mismatches.
required_cols = ['crop_year', 'season', 'crop', 'yield_kg_ha']

# Detect the state column dynamically because the source naming can vary slightly.
state_col = [col for col in df.columns if 'state' in col]

if not state_col:
    raise ValueError("No state column found. Check cleaned.csv")
else:
    state_col = state_col[0]

required_cols.append(state_col)

missing = [col for col in required_cols if col not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

# ==============================
# 3. YEAR NORMALIZATION
# ==============================
# Normalize year so the training scale matches the runtime inference helper.
year_min = df['crop_year'].min()
year_max = df['crop_year'].max()

if year_max == year_min:
    raise ValueError("All crop_year values are same. Normalization will fail.")

df['year_normalized'] = (df['crop_year'] - year_min) / (year_max - year_min)

# ==============================
# 4. ONE-HOT ENCODING
# ==============================
# Expand categorical fields into the one-hot columns expected by inference.
df_encoded = pd.get_dummies(
    df,
    columns=[state_col, 'crop', 'season'],
    prefix=['state', 'crop', 'season']
)

# ==============================
# 5. SPLIT FEATURES & TARGET
# ==============================
# Separate the target from the model inputs before saving the contract.
y = df_encoded['yield_kg_ha']
X = df_encoded.drop(columns=['yield_kg_ha', 'crop_year'])

# ==============================
# 6. CREATE FEATURE CONTRACT
# ==============================
# Store the exact feature order so the Streamlit app can rebuild rows safely.
feature_cols = X.columns.tolist()

with open('feature_columns.json', 'w') as f:
    json.dump(feature_cols, f)

print(f"Contract Created â†’ {len(feature_cols)} features")

# ==============================
# 7. SAVE FINAL DATA
# ==============================
# Persist the fully engineered dataset for later model training and inspection.
df_final = pd.concat([X, y], axis=1)
df_final.to_csv('features.csv', index=False)

print("âœ… features.csv saved")
print("âœ… feature_columns.json saved")